# 03 - Ground Truth and Curated Datasets

This notebook moves from "score a recent trace" to "run a repeatable benchmark."

You will:

1. Model expected responses, assertions, and tool trajectories.
2. Build single-turn and multi-turn scenarios.
3. Invoke the deployed agent with `OnDemandEvaluationDatasetRunner`.
4. Let a polling span collector wait for telemetry instead of using a fixed sleep.

**Estimated time:** 40-60 minutes  
**Creates AWS resources:** No persistent resources; invokes the runtime and evaluation models.  
**Feature status:** Dataset evaluation is public preview as of August 14, 2026.

## 1. Three kinds of reference input

| Reference | Best for | Example |
|---|---|---|
| `expected_response` | Semantic answer comparison | Seattle population and area |
| `assertions` | Requirements that allow multiple valid answers | Must identify both cities |
| `expected_trajectory` | Intended tool sequence | `lookup_city`, then `compare_cities` |

An expected trajectory is useful evidence for session-level evaluation and custom evaluators. Do not assume that the current service exposes a separate built-in "exact trajectory" evaluator unless it appears in your installed CLI or service documentation.

## 2. Load the checked-in scenario file

The service-oriented JSONL format uses `expectedResponse`. The Python SDK model uses `expected_response`, so the loader maps the field explicitly.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from bedrock_agentcore.evaluation import Dataset, PredefinedScenario, Turn

DATASET_PATH = Path("data/city_scenarios.jsonl")
raw_scenarios = [
    json.loads(line)
    for line in DATASET_PATH.read_text().splitlines()
    if line.strip()
]

scenarios = []
for item in raw_scenarios:
    turns = [
        Turn(
            input=turn["input"],
            expected_response=turn.get("expectedResponse"),
        )
        for turn in item["turns"]
    ]
    scenarios.append(
        PredefinedScenario(
            scenario_id=item["scenario_id"],
            turns=turns,
            assertions=item.get("assertions"),
            expected_trajectory=item.get("expected_trajectory"),
        )
    )

dataset = Dataset(scenarios=scenarios)
pd.DataFrame(
    {
        "scenario_id": [item.scenario_id for item in scenarios],
        "turns": [len(item.turns) for item in scenarios],
        "expected_trajectory": [item.expected_trajectory for item in scenarios],
    }
)

## 3. Inspect the multi-turn case

The runner generates one stable session ID per scenario and passes it to every turn. The shared agent uses the Runtime context session ID to isolate in-process conversation state.

In [ ]:
multi_turn = next(
    scenario
    for scenario in scenarios
    if scenario.scenario_id == "multi-turn-follow-up"
)
multi_turn.model_dump()

## 4. Direct ground truth for an existing session

If you already have a session, `ReferenceInputs` is the lightest path. This is useful for investigating a regression before adding it to the full dataset.

In [ ]:
from datetime import timedelta

from bedrock_agentcore.evaluation import EvaluationClient, ReferenceInputs
from src.workshop_utils import (
    RuntimeInvoker,
    load_runtime_info,
    make_session_id,
    wait_for_session_trace,
)

runtime = load_runtime_info()
invoker = RuntimeInvoker(runtime)
direct_session_id = make_session_id("ground-truth")
direct_response = invoker.invoke(
    "Compare Seattle, WA with Portland, OR. Which is denser?",
    direct_session_id,
)
wait_for_session_trace(direct_session_id)

direct_results = EvaluationClient(region_name=runtime.region).run(
    evaluator_ids=[
        "Builtin.GoalSuccessRate",
        "Builtin.Correctness",
        "Builtin.ToolSelectionAccuracy",
    ],
    session_id=direct_session_id,
    agent_id=runtime.runtime_id,
    look_back_time=timedelta(hours=1),
    reference_inputs=ReferenceInputs(
        expected_response=(
            "Seattle is more populous and denser than Portland "
            "in the workshop dataset."
        ),
        assertions=[
            "The response compares both requested cities.",
            "The response identifies Seattle as denser.",
        ],
        expected_trajectory=["compare_cities"],
    ),
)
direct_results

## 5. Configure the dataset runner

The runner has three phases:

1. invoke all scenarios
2. wait for telemetry readiness
3. collect spans and evaluate

`CloudWatchAgentSpanCollector` polls until spans appear. Because it already performs readiness polling, set the runner's fixed `evaluation_delay_seconds` to `0`.

In [ ]:
from bedrock_agentcore.evaluation import (
    CloudWatchAgentSpanCollector,
    EvaluationRunConfig,
    EvaluatorConfig,
    OnDemandEvaluationDatasetRunner,
)

collector = CloudWatchAgentSpanCollector(
    log_group_name=runtime.log_group_name,
    region=runtime.region,
    max_wait_seconds=180,
    poll_interval_seconds=10,
)

run_config = EvaluationRunConfig(
    evaluator_config=EvaluatorConfig(
        evaluator_ids=[
            "Builtin.GoalSuccessRate",
            "Builtin.Correctness",
            "Builtin.Helpfulness",
            "Builtin.ToolSelectionAccuracy",
        ]
    ),
    evaluation_delay_seconds=0,
    max_concurrent_scenarios=3,
)

runner = OnDemandEvaluationDatasetRunner(region=runtime.region)

## 6. Run a small regression pack

Start with three scenarios to control time and cost. After the workflow succeeds, run the full dataset.

In [ ]:
small_dataset = Dataset(scenarios=scenarios[:3])
evaluation_result = runner.run(
    config=run_config,
    dataset=small_dataset,
    agent_invoker=invoker,
    span_collector=collector,
)
evaluation_result.model_dump()

In [ ]:
rows = []
for scenario_result in evaluation_result.scenario_results:
    if scenario_result.status != "COMPLETED":
        rows.append(
            {
                "scenario": scenario_result.scenario_id,
                "evaluator": None,
                "value": None,
                "label": "ERROR",
                "explanation": scenario_result.error,
            }
        )
        continue
    for evaluator_result in scenario_result.evaluator_results:
        for result in evaluator_result.results:
            rows.append(
                {
                    "scenario": scenario_result.scenario_id,
                    "evaluator": evaluator_result.evaluator_id,
                    "value": result.get("value"),
                    "label": result.get("label"),
                    "explanation": result.get("explanation"),
                }
            )

dataset_results_df = pd.DataFrame(rows)
dataset_results_df

## 7. Run the full dataset when ready

This includes:

- a no-tool greeting
- an unknown city that should not be fabricated
- a two-turn follow-up that depends on session continuity

In [ ]:
RUN_FULL_DATASET = False

if RUN_FULL_DATASET:
    full_result = runner.run(
        config=run_config,
        dataset=dataset,
        agent_invoker=invoker,
        span_collector=collector,
    )
    print(full_result.model_dump_json(indent=2))
else:
    print("Set RUN_FULL_DATASET=True after the small regression pack succeeds.")

## 8. Managed datasets through the CLI

The CLI can manage a service dataset and invoke it by name:

```bash
agentcore add dataset \
  --name CityRegression \
  --schema-type AGENTCORE_EVALUATION_PREDEFINED_V1 \
  --description "Stable city-agent regression scenarios"

cp data/city_scenarios.jsonl agentcore/datasets/CityRegression.jsonl

agentcore deploy

agentcore run eval \
  --runtime CityAnalyst \
  --dataset CityRegression \
  --evaluator Builtin.GoalSuccessRate Builtin.Correctness
```

Use the Python runner when your test harness owns invocation and orchestration. Use a managed dataset when teams need versioned, shared scenarios integrated with CLI batch and recommendation workflows.

## 9. Dataset design checklist

A useful regression dataset should include:

- common successful tasks
- known historical failures
- no-tool requests
- invalid or missing parameters
- missing-data behavior
- multi-turn follow-ups
- scenarios that distinguish similar tools

Keep the ground truth stable and review it like code. A large noisy dataset is less useful than a small suite with clear failure ownership.

## 10. Checkpoint

You now have:

- targeted ground truth for one existing session
- a repeatable dataset runner
- single-turn and multi-turn scenarios
- polling-based telemetry readiness

Continue to [04 - Custom Evaluators](04-custom-evaluators.ipynb) to handle failures that the built-in catalog does not express precisely enough.